# 04 Model Diagnostics and Feature Refinement

This notebook reviews why SARIMA/SARIMAX did not beat the seasonal naive benchmark and identifies practical refinements for the next modelling step. It does not add Prophet, simulation or deep learning.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src import model_diagnostics
from src.utils import FIGURES_DIR, TABLES_DIR

TARGET = "nd_mean"

## Run or Load Diagnostics

Run the diagnostics pipeline after baseline and statistical forecasting outputs have been generated.

In [ ]:
outputs = model_diagnostics.run_model_diagnostics(target=TARGET)
for name, table in outputs.items():
    print(name, table.shape)

## Error Summaries by Period

In [ ]:
month_summary = pd.read_csv(TABLES_DIR / "error_summary_by_month.csv")
day_summary = pd.read_csv(TABLES_DIR / "error_summary_by_day_of_week.csv")
regime_summary = pd.read_csv(TABLES_DIR / "error_summary_by_demand_regime.csv")

display(month_summary.sort_values(["month", "mae"]))
display(day_summary.sort_values(["day_of_week", "mae"]))
display(regime_summary.sort_values(["demand_regime", "mae"]))

## Incomplete Day Impact

In [ ]:
incomplete_impact = pd.read_csv(TABLES_DIR / "incomplete_day_forecast_impact.csv")
display(incomplete_impact.head())
if "has_incomplete_day" in incomplete_impact.columns:
    print("Incomplete test days:", int(incomplete_impact["has_incomplete_day"].fillna(False).sum()))

## Residual Autocorrelation

In [ ]:
residual_acf = pd.read_csv(TABLES_DIR / "statistical_residual_autocorrelation.csv")
display(residual_acf.head(20))

## Exogenous Feature Correlations

In [ ]:
feature_corr = pd.read_csv(TABLES_DIR / "exogenous_feature_correlation_with_targets.csv")
display(feature_corr.sort_values(feature_corr.columns[-1], ascending=False))

## Refined Benchmark Comparison

In [ ]:
refined = pd.read_csv(TABLES_DIR / "refined_benchmark_comparison.csv")
display(refined)
best = refined.sort_values("mae").iloc[0]
print(f"Best refined benchmark by MAE: {best['model']} ({best['mae']:.2f})")

## Diagnostic Figures

In [ ]:
for figure_path in [
    FIGURES_DIR / "modelling" / "error_by_month.png",
    FIGURES_DIR / "modelling" / "error_by_demand_regime.png",
    FIGURES_DIR / "modelling" / "statistical_residual_autocorrelation.png",
    FIGURES_DIR / "modelling" / "exogenous_target_correlation.png",
    FIGURES_DIR / "modelling" / "refined_benchmark_comparison.png",
]:
    print(figure_path)
    if figure_path.exists():
        image = plt.imread(figure_path)
        plt.figure(figsize=(14, 6))
        plt.imshow(image)
        plt.axis("off")
        plt.show()

## Interpretation

Seasonal naive remains the benchmark because it currently has the strongest 2025 holdout metrics. SARIMA and SARIMAX should not be described as improvements unless their metrics beat seasonal naive on the same target and test period.

If SARIMAX underperforms, likely explanations include noisy or weakly predictive exogenous variables, insufficient feature engineering, residual autocorrelation that the specification has not captured, and strong year-over-year demand seasonality that the simple benchmark exploits directly. The next modelling step should refine features and validation before adding Prophet or simulation.